# 🧪 TP NumPy – Analyse de données télématiques (4h)

Ce TP vise à pratiquer **NumPy** sur un jeu de données réel issu de la télématique.

⚠️ **Consignes** :
- Utiliser **NumPy autant que possible** (éviter les boucles Python)
- Les réponses doivent être justifiées lorsque demandé
- Aucune cellule de correction n’est fournie


## Contexte

Chaque ligne du fichier correspond à une position GPS envoyée par un boîtier embarqué.
Les données contiennent des informations de temps, position, vitesse et capteurs.

Objectif : **extraire de l’information utile à partir des données brutes**.

# Partie 1 – Prise en main & structures NumPy (≈ 45 min)

### Q1. Chargement des données
1. Charger le fichier CSV avec pandas.
2. Extraire les colonnes numériques principales (`latitude`, `longitude`, `speed`, `altitude`, `accuracy`) sous forme de tableaux NumPy.
3. Vérifier le `dtype` et la `shape` de chaque tableau.

### Q2. Nettoyage minimal
1. Identifier les positions invalides (`valid == 0`).
2. Créer un masque booléen pour conserver uniquement les positions valides.
3. Appliquer ce masque à toutes les colonnes numériques.

### Q3. Mémoire et types
1. Afficher `dtype`, `itemsize` et `nbytes` pour la latitude et la vitesse.
2. Convertir la vitesse en `float32`.
3. Comparer l’empreinte mémoire avant et après conversion.


In [24]:
import pandas as pd
import numpy as np
df = pd.read_csv("telematics_export1.csv")
lat = df["latitude"].to_numpy()
longitude = df["longitude"].to_numpy()
speed = df["speed"].to_numpy()
alt = df["altitude"].to_numpy()

print(df.dtypes, lat.dtype, longitude.dtype, speed.dtype, alt.dtype)
print(df.shape, lat.shape)

#df.head()

id               int64
protocol        object
deviceid         int64
servertime      object
devicetime      object
fixtime         object
valid            int64
latitude       float64
longitude      float64
altitude         int64
speed          float64
course           int64
address        float64
attributes      object
accuracy         int64
network        float64
geofenceids     object
dtype: object float64 float64 float64 int64
(300000, 17) (300000,)


In [25]:
##Question 2

#1)
invalides = (df["valid"] == 0)

#2)
mask = (df["valid"] == 1)

#3)

lat_clean = lat[mask]
longitude_clean = longitude[mask]
speed_clean = speed[mask]
alt_clean = alt[mask]

df_clean = df[mask]
print(df_clean.head())

"""
import numpy as np
lat = np.array([10, 20, 30, 40])
mask = np.array([True, False, True, False])
lat[mask]  # -> array([10, 30])
"""

          id   protocol  deviceid           servertime           devicetime  \
0  476086265  teltonika        97  2026-01-23 03:21:10  2026-01-23 03:21:07   
1  476086264  teltonika        97  2026-01-23 03:21:10  2026-01-23 03:20:49   
2  476086263  teltonika        97  2026-01-23 03:21:10  2026-01-23 03:20:47   
3  476086262  teltonika        97  2026-01-23 03:21:10  2026-01-23 03:20:46   
4  476086261  teltonika        97  2026-01-23 03:21:10  2026-01-23 03:20:33   

               fixtime  valid   latitude  longitude  altitude    speed  \
0  2026-01-23 03:21:07      1  35.760895  -5.817477        26  0.00000   
1  2026-01-23 03:20:49      1  35.760888  -5.817483        26  0.00000   
2  2026-01-23 03:20:47      1  35.760900  -5.817475        25  3.77970   
3  2026-01-23 03:20:46      1  35.760920  -5.817472        24  4.85961   
4  2026-01-23 03:20:33      1  35.761317  -5.817168        16  7.55940   

   course  address                                         attributes  \
0     2

'\nimport numpy as np\nlat = np.array([10, 20, 30, 40])\nmask = np.array([True, False, True, False])\nlat[mask]  # -> array([10, 30])\n'

In [26]:
print(lat_clean.dtype,lat_clean.itemsize,lat_clean.nbytes, lat_clean.shape)
print(lat.dtype,lat.itemsize,lat.nbytes, lat.shape)

print(speed_clean.dtype,speed_clean.itemsize,speed_clean.nbytes, speed_clean.shape)
print(speed.dtype,speed.itemsize,speed.nbytes, speed.shape)

#2 Conversion en float32

lat_clean_float32 = lat_clean.astype(np.float32)
speed_clean_float32 = speed_clean.astype(np.float32)

#3) Comparaison des tailles en mémoire

print(lat_clean_float32.dtype,lat_clean_float32.itemsize,lat_clean_float32.nbytes, lat_clean_float32.shape)
print(speed_clean_float32.dtype, speed_clean_float32.itemsize, speed_clean_float32.nbytes, speed_clean_float32.shape)

#4) comparaison dfs
#print("Types df:", df.dtypes)
#print("Types df_clean:", df_clean.dtypes)

print("Taille df:", df.shape)
print("Taille df_clean:", df_clean.shape)

#print("Mémoire df:", df.memory_usage(deep=True))
#print("Mémoire df_clean:", df_clean.memory_usage(deep=True))


float64 8 2378144 (297268,)
float64 8 2400000 (300000,)
float64 8 2378144 (297268,)
float64 8 2400000 (300000,)
float32 4 1189072 (297268,)
float32 4 1189072 (297268,)
Taille df: (300000, 17)
Taille df_clean: (297268, 17)


# Partie 2 – Calculs vectorisés & géométrie

### Q4. Vitesse et mouvement
1. Calculer la vitesse moyenne, minimale et maximale.
2. Identifier les positions où la vitesse dépasse 90 km/h.
3. Compter le nombre de positions concernées.

### Q5. Distances vectorisées
On considère un point de référence (dépôt) :
- `lat_ref = 34.35`
- `lon_ref = -6.51`

1. Calculer la distance euclidienne (approximative) entre chaque position et le dépôt.
2. Trouver la distance maximale atteinte.
3. Identifier la position la plus proche du dépôt.

### Q6. Broadcasting
1. Construire un tableau `(N, 2)` contenant `(latitude, longitude)`.
2. Soustraire le point de référence en une seule opération NumPy.
3. Expliquer pourquoi le broadcasting fonctionne.


In [27]:
#1)
vit_moy = speed_clean.mean()
vit_min = speed_clean.min()
vit_max = speed_clean.max()

print(vit_moy,vit_min,vit_max)

#2) vitesse > 90
speed90_clean = speed_clean > 90

#3) compter les posittions
print(speed90_clean.sum())

17.270244728043377 0.0 102.592
8


In [28]:
## Question 5
#from math import sqrt marche pas il faut np.sqrt
lat_ref = 34.35
long_ref = -6.51

# 1)
dist_lat_clean = lat_clean - lat_ref
dist_long_clean = longitude_clean - long_ref

dist_clean = np.sqrt(dist_lat_clean**2 +dist_long_clean**2)
print(dist_clean[:5]) #: → signifie « de … à … »
# 2)
print(dist_long_clean.max())
# 3)
indice_min = np.argmin(dist_clean)
print(indice_min)


[1.57169124 1.57168228 1.57169639 1.57171589 1.57220556]
1.4014767
293789


In [29]:
#Question 6

# 1) tableau (N,2) des positions
pos = np.column_stack((lat_clean, longitude_clean)) #() tuples, [] listes, {} dicts
print(pos.shape)

# 2) soustraction du point de référence en une seule opération
point_ref = np.array([lat_ref, long_ref])
print(point_ref.shape)

pos_Minus_point_ref = pos - point_ref
print(pos_Minus_point_ref.shape)



(297268, 2)
(2,)
(297268, 2)


# Partie 3 – Masques booléens & filtrage 

### Q7. Masques avancés
1. Créer un masque correspondant aux positions : vitesse > 80 km/h **ET** altitude < 20 m.
2. Combien de positions respectent ces critères ?
3. Extraire les coordonnées correspondantes.

### Q8. Analyse conditionnelle
1. Calculer la vitesse moyenne sur l’ensemble des données.
2. Calculer la vitesse moyenne lorsque la vitesse > 60 km/h.
3. Comparer et commenter.

### Q9. Indexation
1. Extraire les 10 premières positions où la vitesse dépasse 100 km/h.
2. Extraire 10 positions aléatoires.
3. Comparer indexation booléenne et fancy indexing.


In [ ]:
#Question 7

# Erreurs rencontrées:
# 1) (speed_clean > 70) & (alt < 20) /!\ aux parentheses l'ordre n'est pas intuitif
# 2) mismatch de taille, il faut prendre alt_CLEAN aussi
# 3) raccourci même mot: ctrl shift L

msk_SpeedO70_AltU20 = (speed_clean > 70) & (alt_clean < 20)
print(msk_SpeedO70_AltU20[:5])

# 2) on applique le masque sur df_clean et le stock dans df_clean2, puis compte avec deux méthodes
df_clean2 = df_clean.loc[msk_SpeedO70_AltU20] # .loc pour expliciter qu'on travaille sur les lignes

nb_respect_msk_SO70_AU20 = df_clean2.shape[0] #option 1: df.shape -> tuple (nb_lignes, nb_colonnes)
print(nb_respect_msk_SO70_AU20)

nb_respect_msk_SO70_AU20v2 = msk_SpeedO70_AltU20.sum() #option 2: compte le nb de true
print(nb_respect_msk_SO70_AU20v2)

# 3) extraire les coordonnées correspondantes
coords = df_clean2[["latitude","longitude"]]
coords_np = df_clean2[["latitude","longitude"]].to_numpy()
print(coords_np[:5])

[False False False False False]
209
209
[[34.8148716 -6.2433266]
 [34.8092233 -6.2515883]
 [34.807695  -6.2541383]
 [34.80712   -6.2551049]
 [34.8027716 -6.2633883]]


In [ ]:
#Question 8

#1) vit moy de l'ens des données
print(vit_moy)

# bonus
vit_moy_SO70_AU20 = df_clean2["speed"].mean()
print(vit_moy_SO70_AU20)
#2) vit moyenne > 60

msk_SO60 = speed_clean > 60
speed_clean2 = speed_clean[msk_SO60]

vit_moy_SU60 = speed_clean2.mean()
print(vit_moy_SU60)

# 3) comparaison
# On voit qu'on a 73 en moyenne pour SO70 et 65 pour SO60


17.270244728043377
73.61758325358852
65.67118713971396


# Partie 4 – Statistiques & agrégations

### Q10. Statistiques globales
1. Calculer moyenne, médiane et écart-type de la vitesse.
2. Calculer min / max de l’altitude.
3. Identifier d’éventuelles valeurs aberrantes.

### Q11. Agrégations par axe
1. Construire un tableau `(N, 3)` avec latitude, longitude et vitesse.
2. Calculer la moyenne de chaque colonne.
3. Interpréter ces valeurs comme des *features globales*.

### Q12. Binning
1. Créer des classes de vitesse : `[0–30[, [30–60[, [60–90[, [90+]`.
2. Associer chaque position à une classe.
3. Compter le nombre de positions par classe.
4. Identifier la classe dominante.


# Partie 5 – Tri, partition & cas IA

### Q13. Tri et argsort
1. Trier les positions par vitesse croissante.
2. Identifier les 5 positions les plus rapides.
3. Extraire leurs coordonnées GPS.

### Q14. Tri partiel
1. Utiliser `argpartition` pour trouver les 10 vitesses les plus élevées.
2. Comparer avec un tri complet.
3. Expliquer l’intérêt algorithmique.

### Q15. Mini k-NN
1. Choisir une position GPS comme référence.
2. Calculer les distances à toutes les autres positions.
3. Trouver les 5 positions les plus proches.
4. Interpréter le résultat.


# Partie 6 – Lien IA & Feature Engineering 

### Q16. Construction de features
Créer un vecteur NumPy contenant :
- vitesse moyenne
- vitesse max
- écart-type de la vitesse
- distance maximale au dépôt
- nombre de positions > 90 km/h

### Q17. Interprétation IA
Expliquer comment ces features pourraient être utilisées pour :
- détecter une conduite agressive
- classifier des trajets
- construire un score conducteur
